# Hyperparameter tuning

Hyperparameter tuning can substantially improve a model's performance.
To make this easier, sgtlearn provides a recommended search space for
scikit-learn and Optuna.

To get started, install the tuning extra:

```console
pip install 'sgtlearn[tune]'
```

For this tutorial, we'll use scikit-learn's breast cancer dataset.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split,
)
from sgtlearn import SGTClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42,
)

We offer two ways to use the search space.

## 1. A dictionary for scikit-learn searches

`suggested_params()` returns a dictionary of parameter names and candidate values.

In [2]:
from sgtlearn.tune import suggested_params

space = suggested_params()
space

{'max_depth': [4, 6, 8, 10],
 'num_partitions': [2, 3],
 'inner_max_leaf_nodes': [2, 4, 8, 16, 32, 64],
 'min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'inner_min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'min_impurity_decrease': [0.0, 0.01, 0.1, 1.0, 10.0],
 'inner_min_impurity_decrease': [0.0, 0.001, 0.01, 0.1],
 'branching_penalty': [0.0, 0.01, 0.1, 1.0]}

Pass it directly to `RandomizedSearchCV` to try a fixed number of configurations.

In [3]:
random_search = RandomizedSearchCV(
    SGTClassifier(), param_distributions=space,
    n_iter=10, cv=3, scoring="balanced_accuracy", random_state=42,
)
random_search.fit(X_train, y_train)
print(f"Best CV balanced accuracy: {random_search.best_score_:.3f}")

Best CV balanced accuracy: 0.920


`GridSearchCV` accepts the same dictionary. Select just `max_depth` and
`inner_max_leaf_nodes` to tune these two parameters and leave the others at their defaults.

In [4]:
grid_space = {name: space[name] for name in ["max_depth", "inner_max_leaf_nodes"]}

grid_search = GridSearchCV(
    SGTClassifier(), param_grid=grid_space, cv=3, scoring="balanced_accuracy",
)
grid_search.fit(X_train, y_train)
print(f"Best CV balanced accuracy: {grid_search.best_score_:.3f}")

Best CV balanced accuracy: 0.903


## 2. Suggestions for an Optuna trial

`suggest_params(trial)` uses an Optuna trial to choose parameter values and
returns a dictionary you can pass to the estimator.

In [5]:
import optuna
from sgtlearn.tune import suggest_params

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = suggest_params(trial)
    model = SGTClassifier(**params)
    return cross_val_score(
        model, X_train, y_train, cv=3, scoring="balanced_accuracy",
    ).mean()


study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=10)
print(f"Best CV balanced accuracy: {study.best_value:.3f}")

Best CV balanced accuracy: 0.911


To fit the winner, rebuild its parameters with `FixedTrial`. This also
restores the required `tao_n_runs=10` override if TAO was enabled.

In [6]:
params = suggest_params(optuna.trial.FixedTrial(study.best_trial.params))
model = SGTClassifier(**params).fit(X_train, y_train)
model.predict(X_test[:5])

array([1, 0, 1, 1, 1])

To use TAO or S²GT (bivariate branching), pass `use_tao=True` or
`use_bivariate=True` to either helper, for example
`suggested_params(use_tao=True, use_bivariate=True)` or
`suggest_params(trial, use_tao=True, use_bivariate=True)`. Both default to
`False`; use the same flags when rebuilding an Optuna trial's parameters.

**Both off**

In [7]:
suggested_params(use_tao=False, use_bivariate=False)

{'max_depth': [4, 6, 8, 10],
 'num_partitions': [2, 3],
 'inner_max_leaf_nodes': [2, 4, 8, 16, 32, 64],
 'min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'inner_min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'min_impurity_decrease': [0.0, 0.01, 0.1, 1.0, 10.0],
 'inner_min_impurity_decrease': [0.0, 0.001, 0.01, 0.1],
 'branching_penalty': [0.0, 0.01, 0.1, 1.0]}

**Bivariate on, TAO off**

In [8]:
suggested_params(use_tao=False, use_bivariate=True)

{'max_depth': [4, 6, 8, 10],
 'num_partitions': [2, 3],
 'inner_max_leaf_nodes': [2, 4, 8, 16, 32, 64],
 'min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'inner_min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'min_impurity_decrease': [0.0, 0.01, 0.1, 1.0, 10.0],
 'inner_min_impurity_decrease': [0.0, 0.001, 0.01, 0.1],
 'branching_penalty': [0.0, 0.01, 0.1, 1.0],
 'pairwise_candidates': [0.25, 0.5, 1.0, 2.0],
 'pairwise_penalty': [0.0, 0.01, 0.1, 1.0]}

**TAO on, bivariate off**

In [9]:
suggested_params(use_tao=True, use_bivariate=False)

{'max_depth': [4, 6, 8, 10],
 'num_partitions': [2, 3],
 'inner_max_leaf_nodes': [2, 4, 8, 16, 32, 64],
 'min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'inner_min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'min_impurity_decrease': [0.0, 0.01, 0.1, 1.0, 10.0],
 'inner_min_impurity_decrease': [0.0, 0.001, 0.01, 0.1],
 'branching_penalty': [0.0, 0.01, 0.1, 1.0],
 'tao_lambda': [0.01, 0.001, 0.0001],
 'tao_n_runs': [10]}

**Both on**

In [10]:
suggested_params(use_tao=True, use_bivariate=True)

{'max_depth': [4, 6, 8, 10],
 'num_partitions': [2, 3],
 'inner_max_leaf_nodes': [2, 4, 8, 16, 32, 64],
 'min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'inner_min_samples_leaf': [1, 2, 4, 8, 16, 32],
 'min_impurity_decrease': [0.0, 0.01, 0.1, 1.0, 10.0],
 'inner_min_impurity_decrease': [0.0, 0.001, 0.01, 0.1],
 'branching_penalty': [0.0, 0.01, 0.1, 1.0],
 'tao_lambda': [0.01, 0.001, 0.0001],
 'tao_n_runs': [10],
 'pairwise_candidates': [0.25, 0.5, 1.0, 2.0],
 'pairwise_penalty': [0.0, 0.01, 0.1, 1.0],
 'tao_pair_scale': [1.5, 1.75, 2.0]}